# The Plasm Koans

Each koan below has blanks (`__`) to fill in. Run the setup cell once, then run each koan cell:
it runs the koan in its own sandbox, so every koan starts from the same data, and shows ✔ when
it passes, or what to fill in or fix. Edit a cell and run it again until it passes.

Build the koans first: `dotnet build koans/Plasm.Koans`. The setup cell starts a throwaway
PostgreSQL container, or uses the server named by `PLASM_KOANS_PG`.

In [ ]:
#I "../Plasm.Koans/bin/Debug/net11.0"
#r "Microsoft.Extensions.Logging.Abstractions.dll"
#r "Plasm.dll"
#r "Plasm.Postgres.dll"
#r "Npgsql.dll"
#r "Testcontainers.dll"
#r "Testcontainers.PostgreSql.dll"
#r "Plasm.Koans.dll"
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop
open Plasm.Koans.Notebook

start ()

## About records and schemas

Domain types are ordinary immutable F# records and single-case unions. Persistence lives in separate schema values and scalar mappings, and a model is validated once, before any query runs.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koan "A record needs nothing from Plasm" (fun _ ->
    // Product is a plain F# record: no base class, no attributes.
    // Its table and key are described separately, in Db.products.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    let renamed = { widget with Name = "Better widget" }
    widget.Name |> shouldBe __
    renamed.Name |> shouldBe __)

In [ ]:
koanTask "Wrapped keys keep their meaning" (fun k ->
    task {
        // CustomerId wraps the bigint key (Scalar.wrap in Db.scalars).
        let! ada = k.Repo.One(plasm { for c in Db.customers do where (c.Email = "ada@example.com") })
        let ada = (ok ada).Value
        ada.Id |> shouldBe __
    })

In [ ]:
koanTask "Stored values can be rejected" (fun k ->
    task {
        // Quantity can only be built by Quantity.create, which refuses zero.
        Quantity.create 0 |> Result.isOk |> shouldBe __

        // A zero stored in the database is refused on the way in, with its field named.
        let! _ = k.Repo.ExecuteSql $"update order_lines set quantity = 0 where id = 1"
        let! lines = k.Repo.All(Query.from Db.orderLines)

        match lines with
        | Error(DbError.Materialization m) -> m.Property |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "Absent values are options" (fun k ->
    task {
        // Order 4 has no customer (a NULL column) but has a cancellation reason.
        let! order = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId 4L) })
        let order = (ok order).Value
        order.CustomerId |> shouldBe __
        order.CancellationReason |> shouldBe __
    })

In [ ]:
koan "A model reports every mistake at once" (fun _ ->
    // Building a model with no scalar mappings: CustomerId, OrderId,
    // OrderStatus and the rest cannot be stored.
    match Model.build [] [ Db.customers; Db.orders ] with
    | Ok _ -> failwith "expected configuration errors"
    | Error errors ->
        (errors.Length > 1) |> shouldBe __
        errors |> List.exists (fun e -> e.Model = "Order") |> shouldBe __)

## About query values

A query is a value: building, extending and inspecting it never touches the database. Execution happens only at the repository.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koan "Building a query runs nothing" (fun k ->
    let _published = plasm { for o in Db.orders do where (o.Status = Published) }
    // How many commands has this koan sent?
    k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "Extending makes a new query" (fun k ->
    task {
        let all = Query.from Db.orders
        let published = plasm { for o in all do where (o.Status = Published) }
        let! everyOrder = k.Repo.Count all
        let! publishedOrders = k.Repo.Count published
        ok everyOrder |> shouldBe __
        ok publishedOrders |> shouldBe __
    })

In [ ]:
koanTask "Running does not change a query" (fun k ->
    task {
        let drafts = plasm { for o in Db.orders do where (o.Status = Draft) }
        let! first = k.Repo.All drafts
        let! second = k.Repo.All drafts
        (ok first = ok second) |> shouldBe __
        (ok first).Length |> shouldBe __
    })

In [ ]:
koan "A query can explain itself" (fun k ->
    let published = plasm { for o in Db.orders do where (o.Status = Published) }
    let sql = k.Sql published
    // Is the status sent as a parameter?
    sql.Contains "\"status\" = @p0" |> shouldBe __
    k.Commands.Length |> shouldBe __)

In [ ]:
koanTask "Reusable filters are functions" (fun k ->
    task {
        let published (q: Query<Order>) = plasm { for o in q do where (o.Status = Published) }
        let over (amount: decimal) (q: Query<Order>) = plasm { for o in q do where (o.Total > amount) }
        let! n = k.Repo.Count(Query.from Db.orders |> published |> over 50m)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Captured values become parameters" (fun k ->
    task {
        let email = "bob@example.com"
        let byEmail = plasm { for c in Db.customers do where (c.Email = email) }
        // Is the email written into the SQL text?
        (k.Sql byEmail).Contains "bob@example.com" |> shouldBe __
        let! bob = k.Repo.One byEmail
        (ok bob).Value.Name |> shouldBe __
    })

## About filters and ordering

Filters mean what the same F# predicate means. Anything that cannot run in the database is refused, never evaluated in memory.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let count (k: KoanContext) (q: Query<Order>) =
    task {
        let! n = k.Repo.Count q
        return ok n
    }

In [ ]:
koanTask "Independent filters combine" (fun k ->
    task {
        let! n =
            count k (plasm {
                for o in Db.orders do
                    where (o.Status = Published)
                    where (o.Total > 50m)
            })

        n |> shouldBe __
    })

In [ ]:
koanTask "Comparing with None" (fun k ->
    task {
        let withoutCustomer = plasm { for o in Db.orders do where (o.CustomerId = None) }
        (k.Sql withoutCustomer).Contains "IS NULL" |> shouldBe __
        let! n = count k withoutCustomer
        n |> shouldBe __
    })

In [ ]:
koanTask "not keeps F#'s meaning" (fun k ->
    task {
        // Orders that are not Ada's: in F#, an Order with no customer is not Ada's.
        let notAdas = plasm { for o in Db.orders do where (not (o.CustomerId = Some(CustomerId 1L))) }
        let! n = count k notAdas
        n |> shouldBe __
    })

In [ ]:
koanTask "Membership in an empty list" (fun k ->
    task {
        let wanted: OrderStatus list = []
        let! n = count k (plasm { for o in Db.orders do where (List.contains o.Status wanted) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Some code cannot be translated" (fun k ->
    task {
        let shout (s: string) = s.ToUpperInvariant() + "!"
        let! r = k.Repo.All(plasm { for c in Db.customers do where (shout c.Name = "ADA!") })

        let refused =
            match r with
            | Error(DbError.Translation(TranslationError.UnsupportedExpression _)) -> true
            | _ -> false

        refused |> shouldBe __
    })

In [ ]:
koanTask "Pages need an order" (fun k ->
    task {
        let! unordered = k.Repo.All(plasm { for o in Db.orders do take 2 })
        Result.isError unordered |> shouldBe __

        let! page = k.Repo.All(plasm { for o in Db.orders do orderBy o.Total; take 2 })
        ok page |> List.map (fun o -> o.Total) |> shouldBe __
    })

In [ ]:
koanTask "Captured arrays are read each run" (fun k ->
    task {
        let statuses = [| Published |]
        let matching = plasm { for o in Db.orders do where (Array.contains o.Status statuses) }
        let! before = count k matching
        statuses.[0] <- Draft
        let! after = count k matching
        before |> shouldBe __
        after |> shouldBe __
    })

## About reading

The repository runs queries and returns Results. Counting and existence transfer no rows; projections read only the columns they use.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Reading returns typed rows" (fun k ->
    task {
        let! orders = k.Repo.All(plasm { for o in Db.orders do orderBy o.Id })
        let first = (ok orders).Head
        // Total is a decimal; Status is an OrderStatus, not a string.
        first.Total |> shouldBe __
        first.Status |> shouldBe __
    })

In [ ]:
koanTask "Counting transfers no rows" (fun k ->
    task {
        let! n = k.Repo.Count(plasm { for o in Db.orders do where (o.Status = Published) })
        ok n |> shouldBe __
        k.Commands.Head.Contains "COUNT(*)" |> shouldBe __
    })

In [ ]:
koanTask "Asking whether anything exists" (fun k ->
    task {
        let! any = k.Repo.Exists(plasm { for o in Db.orders do where (o.Total > 1000m) })
        ok any |> shouldBe __
    })

In [ ]:
koanTask "Selecting only what you need" (fun k ->
    task {
        let totals = plasm { for o in Db.orders do orderBy o.Id; select (o.Id, o.Total) }
        let! rows = k.Repo.All totals
        (ok rows).Head |> shouldBe __
        // Is the status column read?
        k.Commands.Head.Contains "\"status\"" |> shouldBe __
    })

In [ ]:
koanTask "Leaving the database is explicit" (fun k ->
    task {
        let! published = k.Repo.All(plasm { for o in Db.orders do where (o.Status = Published) })
        // From here on it is an ordinary F# list, in memory.
        ok published |> List.sumBy (fun o -> o.Total) |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "One means at most one" (fun k ->
    task {
        let! largest = k.Repo.One(plasm { for o in Db.orders do orderByDescending o.Total; take 1 })
        (ok largest |> Option.map (fun o -> o.Total)) |> shouldBe __
        let! none = k.Repo.One(plasm { for o in Db.orders do where (o.Total > 1000m) })
        ok none |> shouldBe __
    })

## About relationships

Relationships are loaded only when asked, in batched commands, and never by reading a field.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let byId (orders: Order list) (id: int64) = orders |> List.find (fun o -> o.Id = OrderId id)

In [ ]:
koanTask "Not loaded is not empty" (fun k ->
    task {
        let! plain = k.Repo.All(Query.from Db.orders)
        (byId (ok plain) 1L).Lines |> shouldBe __

        // Order 4 has no lines: once loaded, that is an empty list.
        let! loaded = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        (byId (ok loaded) 4L).Lines |> shouldBe __
    })

In [ ]:
koanTask "Reading a field never queries" (fun k ->
    task {
        let! orders = k.Repo.All(Query.from Db.orders)

        for o in ok orders do
            for _ in 1..3 do
                o.Lines |> ignore

        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Loads are batched across parents" (fun k ->
    task {
        // Four Orders, and their Lines.
        let! _ = k.Repo.All(plasm { for o in Db.orders do preload o.Lines })
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Siblings load separately" (fun k ->
    task {
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Lines; preload o.Payments })
        let first = byId (ok orders) 1L
        k.Commands.Length |> shouldBe __
        (first.Lines |> Assoc.get |> List.length, first.Payments |> Assoc.get |> List.length) |> shouldBe __
    })

In [ ]:
koanTask "Loads nest" (fun k ->
    task {
        let lines = plasm { for l in Db.orderLines do orderBy l.Id; preload l.Product }
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Lines lines })
        let firstLine = (byId (ok orders) 1L).Lines |> Assoc.get |> List.head
        (firstLine.Product |> Assoc.get).Name |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "A child filter filters children" (fun k ->
    task {
        let succeeded = plasm { for p in Db.payments do where (p.Status = Succeeded) }
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Payments succeeded })
        (ok orders).Length |> shouldBe __
        (byId (ok orders) 3L).Payments |> Assoc.get |> List.length |> shouldBe __
    })

In [ ]:
koanTask "Filtering parents through children" (fun k ->
    task {
        // Order 3 has two payments, one of them succeeded.
        let paid = plasm { for o in Db.orders do where (Rel.any (fun p -> p.Status = Succeeded) o.Payments); orderBy o.Id }
        let! orders = k.Repo.All paid
        ok orders |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "At most n per parent" (fun k ->
    task {
        let latest = plasm { for p in Db.payments do orderByDescending p.CreatedAt; limitPerParent 1 }
        let! orders = k.Repo.All(plasm { for o in Db.orders do preload o.Payments latest })
        (byId (ok orders) 3L).Payments |> Assoc.get |> List.map (fun p -> p.Status) |> shouldBe __
    })

In [ ]:
koanTask "Loading onto values you hold" (fun k ->
    task {
        let! orders = k.Repo.All(Query.from Db.orders)
        let originals = ok orders
        let! withLines = k.Repo.Preload(plasm { for o in Db.orders do preload o.Lines }, originals)
        (byId originals 1L).Lines |> shouldBe __
        (byId (ok withLines) 1L).Lines |> Assoc.get |> List.length |> shouldBe __
    })

## About changesets

Changes are values that can be validated and inspected before anything is written.

In [ ]:
open System.Collections.Generic
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let ada = { New.customer "Ada" "ada@example.com" with Id = CustomerId 1L }

In [ ]:
koan "A changeset leaves its input alone" (fun _ ->
    let cs = ada |> Changeset.edit (fun c -> { c with Name = "Ada L." })
    ada.Name |> shouldBe __
    cs.Changes |> List.map fst |> shouldBe __)

In [ ]:
koan "Unchanged and cleared differ" (fun _ ->
    let cancelled = { New.order None Cancelled 5m with Id = OrderId 4L; CancellationReason = Some "duplicate" }
    let cs = cancelled |> Changeset.edit (fun o -> { o with CancellationReason = None })
    // Every other field is unchanged, so it is not a change at all.
    cs.Changes |> shouldBe __)

In [ ]:
koan "Errors can be inspected" (fun k ->
    let cs =
        Changeset.insert (New.customer "" "not an email")
        |> Changeset.validateLength ((fun c -> c.Name), min = 1)
        |> Changeset.validateFormat ((fun c -> c.Email), "^[^@\\s]+@[^@\\s]+$")

    cs.IsValid |> shouldBe __
    cs.Errors |> List.map (fun e -> e.Field) |> shouldBe __
    k.Commands.Length |> shouldBe __)

In [ ]:
koan "Input cannot touch protected fields" (fun _ ->
    let form = readOnlyDict [ "name", "Mallory"; "email", "ada@new.example.com"; "id", "99" ]
    let cs = Changeset.update ada |> Changeset.cast (form, fun c -> c.Email)
    cs.Changes |> List.map fst |> shouldBe __)

In [ ]:
koanTask "Invalid changes are never written" (fun k ->
    task {
        let invalid = Changeset.insert (New.customer "" "dan@example.com") |> Changeset.validateLength ((fun c -> c.Name), min = 1)
        let! r = k.Repo.Insert invalid

        let isInvalid =
            match r with
            | Error(WriteError.Invalid _) -> true
            | _ -> false

        isInvalid |> shouldBe __
        let! n = k.Repo.Count(Query.from Db.customers)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Constraints become field errors" (fun k ->
    task {
        let again =
            Changeset.insert (New.customer "Ada again" "ada@example.com")
            |> Changeset.uniqueConstraint ((fun c -> c.Email), "customers_email_key")

        let! r = k.Repo.Insert again

        match r with
        | Error(WriteError.Invalid cs) -> cs.Errors |> List.map (fun e -> e.Field) |> shouldBe __
        | other -> failwithf "expected invalid changes, got %A" other
    })

## About writes

Inserts, updates and deletes take changesets and return new values or typed errors.

In [ ]:
open System
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let order (k: KoanContext) (id: int64) =
    task {
        let! o = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId id) })
        return (ok o).Value
    }

In [ ]:
koanTask "Inserting returns the stored record" (fun k ->
    task {
        let sprocket = New.product "SPROCKET" "Sprocket"
        let! stored = k.Repo.Insert(Changeset.insert sprocket)
        // The database generated a key (its value depends on earlier inserts).
        ((ok stored).Id <> ProductId 0L) |> shouldBe __
        sprocket.Id |> shouldBe __
    })

In [ ]:
koanTask "Updates write only the changes" (fun k ->
    task {
        let! draft = order k 2L
        let! _ = k.Repo.Update(draft |> Changeset.edit (fun o -> { o with Status = Published }))
        let update = List.last k.Commands
        // Which columns does the UPDATE set?
        update.Contains "\"status\" = " |> shouldBe __
        update.Contains "\"total\" = " |> shouldBe __
    })

In [ ]:
koanTask "Stale updates are refused" (fun k ->
    task {
        let! first = order k 1L
        let! _ = k.Repo.Update(first |> Changeset.edit (fun o -> { o with Total = 31m }))
        // `first` still has the old version.
        let! again = k.Repo.Update(first |> Changeset.edit (fun o -> { o with Total = 32m }))

        let stale =
            match again with
            | Error(WriteError.Stale _) -> true
            | _ -> false

        stale |> shouldBe __
    })

In [ ]:
koanTask "Infrastructure is not a domain error" (fun k ->
    task {
        // orders_total_check is not declared on the changeset, so it stays a database failure.
        let! r = k.Repo.Insert(Changeset.insert (New.order None Draft -1m))

        match r with
        | Error(WriteError.Failed f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

In [ ]:
koan "Each call takes its own kind of changeset" (fun k ->
    let refusal =
        try
            k.Repo.Update(Changeset.insert (New.product "X" "X")) |> ignore
            ""
        with :? ArgumentException as e ->
            e.Message

    refusal.StartsWith "repo.Update was given a changeset for insert" |> shouldBe __)

## About child writes

Writing a parent with its children is atomic, and removing children always needs a stated policy and scope.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let withLines (k: KoanContext) (id: int64) =
    task {
        let! o = k.Repo.One(plasm { for o in Db.orders do where (o.Id = OrderId id); preload o.Lines })
        return (ok o).Value
    }

let linesOf (k: KoanContext) (id: int64) =
    task {
        let! n = k.Repo.Count(plasm { for l in Db.orderLines do where (l.OrderId = OrderId id) })
        return ok n
    }

In [ ]:
koanTask "Omitted children are untouched" (fun k ->
    task {
        let! o = withLines k 1L
        let! _ = k.Repo.Update(o |> Changeset.edit (fun o -> { o with Total = 40m }))
        let! n = linesOf k 1L
        n |> shouldBe __
    })

In [ ]:
koanTask "Children are matched by identity" (fun k ->
    task {
        let! o = withLines k 1L
        let first = o.Lines |> Assoc.get |> List.head
        let repriced = first |> Changeset.edit (fun l -> { l with UnitPrice = 9m })
        let added = Changeset.insert (New.line (ProductId 3L) 1 2.5m)
        let! _ = k.Repo.Update(Changeset.update o |> Changeset.putChildren ((fun o -> o.Lines), [ repriced; added ]))
        let! n = linesOf k 1L
        n |> shouldBe __
    })

In [ ]:
koanTask "Removing children needs a policy" (fun k ->
    task {
        let! o = withLines k 1L
        let keepFirst = [ Changeset.update (o.Lines |> Assoc.get |> List.head) ]
        let replace = Changeset.update o |> Changeset.replaceChildren ((fun o -> o.Lines), ReplaceScope.allChildren, keepFirst)

        // Db.orders declares no OnReplace policy for Lines: removing one is refused.
        let! refused = k.Repo.Update replace
        Result.isError refused |> shouldBe __

        // Db.ordersDeletingLines declares OnReplace.Delete.
        let! allowed = (k.RepoWith Db.modelDeletingLines).Update replace
        ok allowed |> ignore
        let! n = linesOf k 1L
        n |> shouldBe __
    })

In [ ]:
koanTask "Parent and children succeed together" (fun k ->
    task {
        // The second line's negative price breaks order_lines_price_check.
        let lines = [ Changeset.insert (New.line (ProductId 1L) 1 10m); Changeset.insert (New.line (ProductId 2L) 1 -1m) ]
        let! r = k.Repo.Insert(Changeset.insert (New.order None Draft 9m) |> Changeset.putChildren ((fun o -> o.Lines), lines))
        Result.isError r |> shouldBe __
        let! orders = k.Repo.Count(Query.from Db.orders)
        ok orders |> shouldBe __
    })

In [ ]:
koanTask "A filtered list is not the whole list" (fun k ->
    task {
        let! o = withLines k 1L
        let gadgets = plasm { for l in Db.orderLines do where (l.ProductId = ProductId 2L) }
        // Replace only the Gadget lines (with none); the Widget line is outside the scope.
        let replace = Changeset.update o |> Changeset.replaceChildren ((fun o -> o.Lines), ReplaceScope.matching gadgets, [])
        let! _ = (k.RepoWith Db.modelDeletingLines).Update replace
        let! left = k.Repo.All(plasm { for l in Db.orderLines do where (l.OrderId = OrderId 1L) })
        ok left |> List.map (fun l -> l.ProductId) |> shouldBe __
    })

In [ ]:
koanTask "The same child twice is refused" (fun k ->
    task {
        let! o = withLines k 1L
        let first = o.Lines |> Assoc.get |> List.head
        let twice = [ Changeset.update first; Changeset.update first ]
        let! r = k.Repo.Update(Changeset.update o |> Changeset.putChildren ((fun o -> o.Lines), twice))

        let duplicate =
            match r with
            | Error(WriteError.Invalid cs) -> cs.Errors |> List.exists (fun e -> e.Kind = FieldErrorKind.DuplicateIdentity)
            | _ -> false

        duplicate |> shouldBe __
    })

## About transactions

A transaction is a value made of named steps. Running it is explicit; a failure names its step; retries are opt-in.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let customers (k: KoanContext) =
    task {
        let! n = k.Repo.Count(Query.from Db.customers)
        return ok n
    }

In [ ]:
koanTask "Steps compose with let!" (fun k ->
    task {
        let signUp =
            transaction {
                let! dan = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                let! order = Tx.insert "order" (Changeset.insert (New.order (Some dan.Id) Draft 1m))
                return order.CustomerId = Some dan.Id
            }

        let! r = k.Repo.Transact signUp
        ok r |> shouldBe __
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "A failure names its step" (fun k ->
    task {
        let signUp =
            transaction {
                let! dan = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                // A negative total breaks orders_total_check.
                let! _ = Tx.insert "order" (Changeset.insert (New.order (Some dan.Id) Draft -1m))
                return ()
            }

        let! r = k.Repo.Transact signUp
        let e = err r
        e.Step |> shouldBe __
        e.RolledBack |> shouldBe __
        let! n = customers k
        n |> shouldBe __
    })

In [ ]:
koanTask "An empty transaction is fine" (fun k ->
    task {
        let! r = k.Repo.Transact(transaction { return 42 })
        r |> shouldBe __
    })

In [ ]:
koan "Retries are opt-in" (fun _ ->
    RetryPolicy.None.MaxAttempts |> shouldBe __
    (RetryPolicy.OnSerializationFailure 3).MaxAttempts |> shouldBe __)

In [ ]:
koanTask "Custom steps use the transaction" (fun k ->
    task {
        let audit =
            transaction {
                let! _ =
                    Tx.run "audit" (fun ctx ->
                        task {
                            use cmd = ctx.CreateCommand()
                            cmd.CommandText <- "insert into products(sku, name) values ('AUDIT', 'Audit')"
                            let! _ = cmd.ExecuteNonQueryAsync(ctx.CancellationToken)
                            return (Ok(): Result<unit, string>)
                        })

                return! Tx.fail "stop" "changed my mind"
            }

        let! r = k.Repo.Transact audit
        (err r).Step |> shouldBe __
        let! audited = k.Repo.Count(plasm { for p in Db.products do where (p.Sku = "AUDIT") })
        ok audited |> shouldBe __
    })

In [ ]:
koanTask "Effects outside the database are not rolled back" (fun k ->
    task {
        let emailsSent = ref 0

        let tx =
            transaction {
                let! _ =
                    Tx.run "email" (fun _ ->
                        task {
                            emailsSent.Value <- emailsSent.Value + 1
                            return (Ok(): Result<unit, string>)
                        })

                return! Tx.fail "stop" "changed my mind"
            }

        let! _ = k.Repo.Transact tx
        emailsSent.Value |> shouldBe __
    })

## About joins

Joins return combined rows from one command; left joins give the missing side as None.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "An inner join gives pairs" (fun k ->
    task {
        let pairs =
            plasm {
                for o in Db.orders do
                    join c in Db.customers on (o.CustomerId = Some c.Id)
                    orderBy o.Id
            }

        let! rows = k.Repo.All pairs
        ok rows |> List.map (fun (o, c) -> o.Id, c.Name) |> shouldBe __
    })

In [ ]:
koanTask "A left join keeps every row" (fun k ->
    task {
        let all =
            plasm {
                for o in Db.orders do
                    leftJoin c in Db.customers on (o.CustomerId = Some c.Id)
                    orderBy o.Id
            }

        let! rows = k.Repo.All all
        ok rows |> List.map (fun (o, c) -> o.Id, c |> Option.map (fun c -> c.Name)) |> List.last |> shouldBe __
    })

In [ ]:
koanTask "Join keys are typed" (fun k ->
    task {
        // The keys must have the same type: an optional CustomerId on the Order side.
        let pairs =
            plasm {
                for o in Db.orders do
                    join c in Db.customers on (o.CustomerId = __)
                    orderBy o.Id
            }

        let! rows = k.Repo.Count pairs
        ok rows |> shouldBe 3L
    })

In [ ]:
koanTask "Filters on joined columns run in SQL" (fun k ->
    task {
        let adas =
            plasm {
                for o in Db.orders do
                    join c in Db.customers on (o.CustomerId = Some c.Id)
                    where (c.Name = "Ada")
            }

        (k.Sql adas).Contains "WHERE" |> shouldBe __
        let! n = k.Repo.Count adas
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Joined queries compose" (fun k ->
    task {
        let withCustomers = plasm { for o in Db.orders do join c in Db.customers on (o.CustomerId = Some c.Id); orderBy o.Id }
        let large (q: Query<Order * Customer>) = plasm { for (o, c) in q do where (o.Total > 20m) }
        let! rows = k.Repo.All(large withCustomers)
        ok rows |> List.map (fun (_, c) -> c.Name) |> shouldBe __
    })

## About aggregates and grouping

Aggregates are typed values computed in the database; grouping summarises without transferring rows.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Counting a page counts the page" (fun k ->
    task {
        let! n = k.Repo.Count(plasm { for o in Db.orders do orderBy o.Id; take 2 })
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Sum and average are typed" (fun k ->
    task {
        let! totals = k.Repo.One(plasm { for o in Db.orders do select (Agg.sum o.Total, Agg.avg o.Total) })
        // The four totals are 30.00, 12.50, 99.00 and 5.00.
        (ok totals).Value |> shouldBe __
    })

In [ ]:
koanTask "No rows is explicit" (fun k ->
    task {
        let! r = k.Repo.One(plasm { for o in Db.orders do where (o.Total > 1000m); select (Agg.sum o.Total, Agg.max o.Total) })
        (ok r).Value |> shouldBe __
    })

In [ ]:
koanTask "Grouping by a key" (fun k ->
    task {
        let perCustomer =
            plasm {
                for o in Db.orders do
                    groupBy o.CustomerId
                    orderBy o.CustomerId
                    select (o.CustomerId, Agg.count ())
            }

        let! rows = k.Repo.All perCustomer
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Filtering groups" (fun k ->
    task {
        let repeat =
            plasm {
                for o in Db.orders do
                    groupBy o.CustomerId
                    having (Agg.count () > 1L)
                    select (o.CustomerId, Agg.count ())
            }

        let! rows = k.Repo.All repeat
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Grouping mistakes are refused" (fun k ->
    task {
        // Status is neither grouped nor aggregated.
        let wrong = plasm { for o in Db.orders do groupBy o.CustomerId; select (o.Status, Agg.count ()) }
        let! r = k.Repo.All wrong
        Result.isError r |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

## About many-to-many and has-one

Join-table relationships and one-to-one relationships load and filter like any other; link writes touch only the join table.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let names (tags: Assoc<Tag list>) = tags |> Assoc.get |> List.map (fun t -> t.Name) |> List.sort

let article (k: KoanContext) (id: int64) =
    task {
        let! a = k.Repo.One(plasm { for a in Db.articles do where (a.Id = ArticleId id); preload a.Tags })
        return (ok a).Value
    }

let tag id name = { Id = TagId id; Name = name }

In [ ]:
koanTask "Tags load through the join table" (fun k ->
    task {
        let! articles = k.Repo.All(plasm { for a in Db.articles do orderBy a.Id; preload a.Tags })
        (ok articles).Head.Tags |> names |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "No tags is an empty, loaded list" (fun k ->
    task {
        let! empty = article k 3L
        empty.Tags |> shouldBe __
    })

In [ ]:
koanTask "Filtering through links" (fun k ->
    task {
        let fsharp = plasm { for a in Db.articles do where (Rel.any (fun (t: Tag) -> t.Name = "fsharp") a.Tags); orderBy a.Id }
        let! rows = k.Repo.All fsharp
        ok rows |> List.map (fun a -> a.Title) |> shouldBe __
    })

In [ ]:
koanTask "Replacing links" (fun k ->
    task {
        let! intro = article k 1L
        let! updated = k.Repo.Update(Changeset.update intro |> Changeset.putLinks ((fun a -> a.Tags), [ tag 4L "ops" ]))
        (ok updated).Tags |> names |> shouldBe __
        let! tags = k.Repo.Count(Query.from Db.tags)
        ok tags |> shouldBe __
    })

In [ ]:
koanTask "Adding and removing one link" (fun k ->
    task {
        let! intro = article k 1L

        let change =
            Changeset.update intro
            |> Changeset.addLinks ((fun a -> a.Tags), [ tag 3L "draft" ])
            |> Changeset.removeLinks ((fun a -> a.Tags), [ tag 2L "sql" ])

        let! updated = k.Repo.Update change
        (ok updated).Tags |> shouldBe __
        let! reloaded = article k 1L
        reloaded.Tags |> names |> shouldBe __
    })

In [ ]:
koanTask "At most one related record" (fun k ->
    task {
        let! accounts = k.Repo.All(plasm { for a in Db.accounts do orderBy a.Id; preload a.Profile })
        ok accounts |> List.map (fun a -> a.Profile |> Assoc.get |> Option.map (fun p -> p.Bio)) |> shouldBe __
    })

In [ ]:
koanTask "Filtering through has-one" (fun k ->
    task {
        let writers = plasm { for a in Db.accounts do where (Rel.exists (fun p -> p.Bio.Contains "compiler") a.Profile) }
        let! rows = k.Repo.All writers
        ok rows |> List.map (fun a -> a.Name) |> shouldBe __
    })

## About bulk writes

Bulk updates and deletes are driven by typed queries; inserts take many records in one command.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Updating what a query matches" (fun k ->
    task {
        let drafts = plasm { for o in Db.orders do where (o.Status = Draft) }
        let! changed = k.Repo.Execute(Bulk.update (drafts, fun o -> { o with Status = Published }))
        ok changed |> shouldBe __
        let! published = k.Repo.Count(plasm { for o in Db.orders do where (o.Status = Published) })
        ok published |> shouldBe __
    })

In [ ]:
koanTask "Updating from the current row" (fun k ->
    task {
        let! _ = k.Repo.Execute(Bulk.update (Query.from Db.orders, fun o -> { o with Total = o.Total * 2m }))
        let! total = k.Repo.One(plasm { for o in Db.orders do select (Agg.sum o.Total) })
        (ok total).Value |> shouldBe __
    })

In [ ]:
koanTask "Deleting what a query matches" (fun k ->
    task {
        let failed = plasm { for p in Db.payments do where (p.Status = Failed) }
        let! removed = k.Repo.Execute(Bulk.delete failed)
        ok removed |> shouldBe __
        let! left = k.Repo.Count(Query.from Db.payments)
        ok left |> shouldBe __
    })

In [ ]:
koanTask "Inserting many records" (fun k ->
    task {
        let parts = [ New.product "BOLT" "Bolt"; New.product "NUT" "Nut"; New.product "WASHER" "Washer" ]
        let! inserted = k.Repo.Execute(Bulk.insert parts)
        ok inserted |> List.map (fun p -> p.Sku) |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

In [ ]:
koanTask "Inserting only what is missing" (fun k ->
    task {
        let catalogue = [ New.product "WIDGET" "Widget"; New.product "SPROCKET" "Sprocket" ]
        let! inserted = k.Repo.Execute(Bulk.insertMissing (catalogue, fun p -> p.Sku))
        ok inserted |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "Bulk writes can be inspected" (fun k ->
    task {
        let plan = k.Repo.Explain(Bulk.delete (plasm { for p in Db.payments do where (p.Status = Failed) }))
        (ok plan).Sql.StartsWith "DELETE FROM \"payments\"" |> shouldBe __
        k.Commands.Length |> shouldBe __
    })

## About hand-written SQL

When a query is easier in SQL, write SQL; interpolated values are always parameters and rows still come back typed.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "SQL can return mapped records" (fun k ->
    task {
        let minimum = 50m
        let! big = k.Repo.Sql<Order>($"SELECT * FROM orders WHERE total >= {minimum} ORDER BY id")
        ok big |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "SQL can return tuples and scalars" (fun k ->
    task {
        let! names = k.Repo.Sql<string>($"SELECT name FROM customers ORDER BY name")
        ok names |> shouldBe __
        let! perStatus = k.Repo.Sql<string * int64>($"SELECT status, count(*) FROM orders GROUP BY status ORDER BY status")
        ok perStatus |> List.head |> shouldBe __
    })

In [ ]:
koanTask "Interpolation is parameters" (fun k ->
    task {
        let name = "Ada'; drop table customers; --"
        let! found = k.Repo.Sql<Customer>($"SELECT * FROM customers WHERE name = {name}")
        ok found |> shouldBe __
        (List.last k.Commands).Contains "@p0" |> shouldBe __
    })

In [ ]:
koanTask "A fragment inside a typed query" (fun k ->
    task {
        let startingWithA = plasm { for c in Db.customers do where (Raw.fragment<bool> "{0} ILIKE {1}" (c.Name, "a%")) }
        let! rows = k.Repo.All startingWithA
        ok rows |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "Commands return affected rows" (fun k ->
    task {
        let! n = k.Repo.ExecuteSql $"update orders set status = 'cancelled' where status = 'published'"
        ok n |> shouldBe __
    })

## About prepared queries

A hot query can be built and planned once and bound to new arguments on each call.

In [ ]:
open System
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let ordersOf =
    Query.prepare (fun (customer: Arg<CustomerId>) ->
        plasm {
            for o in Db.orders do
                where (o.CustomerId = Some customer.Value)
                orderBy o.Id
        })

In [ ]:
koanTask "Built once, run many times" (fun k ->
    task {
        let! adas = k.Repo.Count(ordersOf (CustomerId 1L))
        let! bobs = k.Repo.Count(ordersOf (CustomerId 2L))
        (ok adas, ok bobs) |> shouldBe __
        // Did the two counts send the same SQL text?
        (k.Commands.[0] = k.Commands.[1]) |> shouldBe __
    })

In [ ]:
koan "Arguments are parameters" (fun k ->
    let sql = k.Sql(ordersOf (CustomerId 42L))
    sql.Contains "42" |> shouldBe __)

In [ ]:
koanTask "Paging by arguments" (fun k ->
    task {
        let page =
            Query.prepare2 (fun (size: Arg<int>) (offset: Arg<int>) ->
                plasm {
                    for o in Db.orders do
                        orderBy o.Id
                        skip offset.Value
                        take size.Value
                })

        let! second = k.Repo.All(page 2 2)
        ok second |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Prepared queries still compose" (fun k ->
    task {
        let publishedOf c = plasm { for o in ordersOf c do where (o.Status = Published) }
        let! n = k.Repo.Count(publishedOf (CustomerId 1L))
        ok n |> shouldBe __
    })

In [ ]:
koan "Arguments are read only inside the query" (fun _ ->
    let refused =
        try
            Query.prepare (fun (customer: Arg<CustomerId>) ->
                let id = customer.Value // read while the query is being built
                plasm { for o in Db.orders do where (o.CustomerId = Some id) })
            |> ignore

            false
        with :? InvalidOperationException ->
            true

    refused |> shouldBe __)

In [ ]:
koanTask "Prepared load queries" (fun k ->
    task {
        let paymentsWith =
            Query.prepare (fun (status: Arg<PaymentStatus>) -> plasm { for p in Db.payments do where (p.Status = status.Value) })

        let! orders = k.Repo.All(plasm { for o in Db.orders do where (o.Id = OrderId 3L); preload o.Payments (paymentsWith Failed) })
        (ok orders).Head.Payments |> Assoc.get |> List.map (fun p -> p.Status) |> shouldBe __
    })

## About failures

Expected failures are values you can match on; Plasm's error cases are always qualified, so your domain keeps its own names.

In [ ]:
open System.Threading.Tasks
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

/// Your own union may use the names Plasm uses for its errors.
type Problem =
    | Database
    | Translation

/// A domain error a transaction step can fail with.
type StockError = OutOfStock of sku: string

In [ ]:
koanTask "Failures are values" (fun k ->
    task {
        let kept = ref Unchecked.defaultof<Repo>

        do!
            k.Repo.Sandbox(fun inner ->
                kept.Value <- inner
                Task.FromResult())

        // The sandbox has ended; its repository returns an error rather than throwing.
        let! r = kept.Value.All(Query.from Db.customers)

        match r with
        | Error(DbError.Database f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database error, got %A" other
    })

In [ ]:
koan "Error cases are qualified" (fun _ ->
    let classify (e: DbError) =
        match e with
        | DbError.Database _ -> Database
        | _ -> Translation

    classify (DbError.Translation(TranslationError.InvalidQuery "example")) |> shouldBe __)

In [ ]:
koanTask "Unreadable rows are named" (fun k ->
    task {
        let! _ = k.Repo.ExecuteSql $"update orders set status = 'lost' where id = 1"
        let! r = k.Repo.All(Query.from Db.orders)

        match r with
        | Error(DbError.Materialization m) -> (m.Model, m.Property) |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "SQL mismatches name the column" (fun k ->
    task {
        // Customer needs an email column, and this query does not return one.
        let! r = k.Repo.Sql<Customer>($"SELECT id, name FROM customers")

        match r with
        | Error e -> (string e).Contains "Email" |> shouldBe __
        | Ok rows -> failwithf "expected an error, got %A" rows
    })

In [ ]:
koanTask "Too many rows is about the data" (fun k ->
    task {
        let! r = k.Repo.One(plasm { for o in Db.orders do where (o.Status = Published) })
        r |> shouldBe __
    })

In [ ]:
koanTask "Rejections keep their type" (fun k ->
    task {
        let! r = k.Repo.Transact(transaction { return! Tx.fail "check stock" (OutOfStock "GIZMO") })

        match (err r).Failure with
        | StepFailure.Rejected rejection -> rejection.As<StockError>() |> shouldBe __
        | other -> failwithf "expected a rejection, got %A" other
    })

In [ ]:
koan "Every union is qualified" (fun _ ->
    let cs = Changeset.insert (New.customer "Eve" "eve@example.com")
    cs.Action |> shouldBe __)

## About testing with Plasm

Test against the real database inside sandboxes that always roll back, and check the model against the schema at start-up.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let customers (repo: Repo) =
    task {
        let! n = repo.Count(Query.from Db.customers)
        return ok n
    }

In [ ]:
koanTask "A sandbox rolls everything back" (fun k ->
    task {
        let! inside =
            k.Repo.Sandbox(fun repo ->
                task {
                    let! _ = repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
                    return! customers repo
                })

        let! after = customers k.Repo
        (inside, after) |> shouldBe __
    })

In [ ]:
koanTask "Transactions keep their meaning in a sandbox" (fun k ->
    task {
        let failing =
            transaction {
                let! _ = Tx.insert "customer" (Changeset.insert (New.customer "Dan" "dan@example.com"))
                return! Tx.fail "stop" "changed my mind"
            }

        let! _ = k.Repo.Transact failing
        let! n = customers k.Repo
        n |> shouldBe __
    })

In [ ]:
koanTask "A database error does not end a sandbox" (fun k ->
    task {
        // Every koan already runs in a sandbox: k.Repo is one.
        let! duplicate = k.Repo.Insert(Changeset.insert (New.customer "Ada again" "ada@example.com"))
        Result.isError duplicate |> shouldBe __
        let! later = k.Repo.Insert(Changeset.insert (New.customer "Dan" "dan@example.com"))
        Result.isOk later |> shouldBe __
    })

In [ ]:
koanTask "The model can be checked against the database" (fun k ->
    task {
        let! issues = k.Repo.CheckSchema()
        issues |> shouldBe __
    })

In [ ]:
koanTask "Drift is reported precisely" (fun k ->
    task {
        // DDL is transactional in PostgreSQL, so the sandbox undoes this too.
        let! _ = k.Repo.ExecuteSql $"alter table orders drop column cancellation_reason"
        let! issues = k.Repo.CheckSchema()
        let issue = (ok issues).Head
        (issue.Kind, issue.Model, issue.Field) |> shouldBe __
    })

## About query pipelines

Every plasm { } operation has a pipeline form; both produce the same query.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koan "Pipelines and expressions agree" (fun k ->
    let expression = plasm { for o in Db.orders do where (o.Status = Published); orderBy o.Id }

    let pipeline =
        Query.from Db.orders
        |> Query.where (fun o -> o.Status = Published)
        |> Query.orderBy (fun o -> o.Id)

    (k.Sql pipeline = k.Sql expression) |> shouldBe __)

In [ ]:
koanTask "Pipelines join as often as you like" (fun k ->
    task {
        let gadgetOrders =
            Query.from Db.orders
            |> Query.join (Db.orderLines, (fun o -> o.Id), (fun l -> l.OrderId))
            |> Query.join (Db.products, (fun (_, l) -> l.ProductId), (fun p -> p.Id))
            |> Query.where (fun ((_, _), p) -> p.Sku = "GADGET")
            |> Query.orderBy (fun ((o, _), _) -> o.Id)

        let! rows = k.Repo.All gadgetOrders
        ok rows |> List.map (fun ((o, l), _) -> o.Id, l.Quantity.Value) |> shouldBe __
    })